# Learn `compas_ifc` and `compas_viewer`

In this lesson we will inspect an IFC model, query its elements, export a filtered geometry selection, and open that geometry in `compas_viewer`.

The notebook is intentionally split into two ideas: **IFC meaning and data** come from `compas_ifc`; **interactive geometry inspection** comes from COMPAS and `compas_viewer`.

## 0. Install the lesson dependencies

Run the next cell once in the Jupyter environment used for this lesson. It installs the current COMPAS 2.x-oriented packages used below. If your course environment is already managed, skip the cell and verify the imports in the environment-check cell.

- `compas_ifc` — read, query, and export IFC models
- `compas` — COMPAS geometry and datastructures
- `compas_viewer` — desktop visualization
- `jupyter` — notebook runtime

In [ ]:
%pip install compas_ifc compas compas_viewer jupyter

## 1. Choose an IFC file

Put `building.ifc` beside this notebook, or change `IFC_PATH`. The notebook does not download or create a model for you: students should work with a known IFC file and record where it came from.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

IFC_PATH = Path("building.ifc")
OUTPUT_DIR = Path("exports")
OUTPUT_DIR.mkdir(exist_ok=True)

if not IFC_PATH.exists():
    raise FileNotFoundError(
        f"Place an IFC file at {IFC_PATH.resolve()} or change IFC_PATH."
    )

print(f"Using: {IFC_PATH.resolve()}")

: 

## 2. Record the environment

A reproducible notebook records the interpreter and package versions next to its outputs.

In [ ]:
import compas
import compas_ifc

print("Python:", sys.version)
print("COMPAS:", getattr(compas, "__version__", "unknown"))
print("compas_ifc:", getattr(compas_ifc, "__version__", "unknown"))
try:
    import compas_viewer
    print("compas_viewer:", getattr(compas_viewer, "__version__", "importable"))
except ImportError as error:
    print("compas_viewer is not importable in this kernel:", error)

## 3. Use the `compas_ifc` CLI from Python

The CLI is useful for learning because each command has one clear job. The `--json` option gives the notebook structured data instead of text intended only for a terminal.

In [ ]:
def run_ifc(*arguments, json_output=True):
    command = [sys.executable, "-m", "compas_ifc", *map(str, arguments)]
    if json_output:
        command.append("--json")
    result = subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )
    return json.loads(result.stdout) if json_output else result.stdout

info = run_ifc("info", IFC_PATH)
print(json.dumps(info, indent=2))

### Explore the spatial hierarchy

A building model is not just a collection of shapes. Its spatial hierarchy connects projects, buildings, storeys, and elements.

In [ ]:
tree = run_ifc("tree", IFC_PATH, "--depth", 3)
print(json.dumps(tree, indent=2))

## 4. Query elements by IFC type

Change `ELEMENT_TYPE` and run this cell again. Try `IfcWall`, `IfcWindow`, `IfcDoor`, or `IfcSlab`.

In [ ]:
ELEMENT_TYPE = "IfcWall"
elements_json = run_ifc("list", IFC_PATH, "--type", ELEMENT_TYPE, "--limit", 10)
print(json.dumps(elements_json, indent=2))

## 5. Use the Python object model

The same model can be opened through `BuildingInformationModel`. Start without geometry when the question is about names, types, relationships, or property sets; this keeps the example focused on BIM data.

In [ ]:
from compas_ifc.bim import BuildingInformationModel

model = BuildingInformationModel(
    str(IFC_PATH),
    load_geometries=False,
    rectify_placements=False,
)

elements = model.get_elements_by_type(ELEMENT_TYPE)
print(f"Found {len(elements)} {ELEMENT_TYPE} elements.")
for element in elements[:5]:
    print(element.global_id, element.name, element.ifc_type)

### Inspect one element's property sets

The wrapper exposes convenient identity fields. The underlying IFC entity provides IFC-specific extensions such as `property_sets`.

In [ ]:
if not elements:
    raise ValueError(f"No {ELEMENT_TYPE} elements were found; try another type.")

first_element = elements[0]
print("Name:", first_element.name)
print("GlobalId:", first_element.global_id)
print("Property sets:")
print(json.dumps(first_element._ifc_entity.property_sets, indent=2, default=str))

## 6. Export the filtered geometry

Now we ask `compas_ifc` for geometry for the selected IFC class. The OBJ is a geometry handoff: semantic IFC data stays in the model, while the exported mesh is convenient for COMPAS visualization.

In [ ]:
OBJ_PATH = OUTPUT_DIR / f"{ELEMENT_TYPE.lower()}.obj"
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "compas_ifc",
        "export",
        str(IFC_PATH),
        "--type",
        ELEMENT_TYPE,
        "--to",
        str(OBJ_PATH),
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(f"Exported: {OBJ_PATH.resolve()}")
if result.stdout.strip():
    print(result.stdout)

## 7. Load the result as COMPAS geometry

A COMPAS `Mesh` gives us geometry and topology methods independent of the original IFC query.

In [ ]:
from compas.datastructures import Mesh

mesh = Mesh.from_obj(str(OBJ_PATH))
print("Vertices:", mesh.number_of_vertices())
print("Edges:", mesh.number_of_edges())
print("Faces:", mesh.number_of_faces())

MESH_JSON_PATH = Path("selected_geometry.json")
mesh.to_json(str(MESH_JSON_PATH))
print(f"Serialized for the viewer: {MESH_JSON_PATH.resolve()}")

## 8. Open `compas_viewer` without blocking the notebook

The viewer code lives in `viewer_detached.py` so students can read it as a small standalone COMPAS program. The notebook starts it in a separate process.

In [ ]:
VIEWER_SCRIPT = Path("viewer_detached.py")
if not VIEWER_SCRIPT.exists():
    raise FileNotFoundError(VIEWER_SCRIPT)

viewer_process = subprocess.Popen(
    [sys.executable, str(VIEWER_SCRIPT.resolve())],
    cwd=str(Path.cwd()),
    start_new_session=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Viewer launched in a separate process. Close its window when finished.")
print("Process id:", viewer_process.pid)

## Exercises

1. Change `ELEMENT_TYPE` to `IfcWindow` and rerun from the query cell.
2. Add `--where`, for example `"OverallHeight>2"`, to a CLI query.
3. Find a storey with `run_ifc("list", IFC_PATH, "--type", "IfcBuildingStorey")`, then use `--in` to query only elements contained in it.
4. Change `facecolor` or `linecolor` in `viewer_detached.py` using a COMPAS `Color` object.
5. Explain why the IFC element's `property_sets` and the exported COMPAS mesh are useful for different questions.